In [ ]:
# Pacotes
import ee
import geemap
import geopandas as gpd
import xarray as xr
import os

In [7]:
# 1. Inicialização
ee.Initialize(
    project='ee-gabriel-495521',
    opt_url='https://earthengine-highvolume.googleapis.com',
)

# 2. Definir a Geometria (Baixo São Francisco)
path_json = "../data/bacias_meso_SF.geojson"
gdf = gpd.read_file(path_json)
gdf_menor = gdf[gdf['nm_mesoRH'] == 'Baixo São Francisco']
geom_ee = geemap.geopandas_to_ee(gdf_menor)
area = geom_ee.geometry()

# 3. Definir Período de Interesse
start = "2000-01-01" # Ajustado para bater com a sua série do MapBiomas (2000-2024)
end = "2024-03-20"



In [8]:
# 4. Função de Rescalonamento Otimizada para pr e ET simultaneamente
def scale_xavier(img):
    # Aplica os fatores de escala específicos para cada banda
    pr_scaled = img.select('pr').multiply(0.00686666).add(225.0)
    et_scaled = img.select('ET').multiply(0.05118110).add(0.0)
    
    # Substitui as bandas originais pelas reescalonadas e mantém as propriedades (data)
    return img.addBands([pr_scaled, et_scaled], overwrite=True)\
              .copyProperties(img, img.propertyNames())

# 5. Carregar e Filtrar a Coleção BR-DWGD (Xavier)
colecao_xavier = (
    ee.ImageCollection("projects/ee-alexandrexavier/assets/BR-DWGD")
    .filterBounds(area)
    .filterDate(start, end)
    .select(['pr', 'ET']) # Puxa só o que importa
    .map(scale_xavier)
)

print("Baixando o cubo de dados (Precipitação e ETP) para a memória...")

Baixando o cubo de dados (Precipitação e ETP) para a memória...


In [12]:

from xee import helpers

aoi = gdf_menor.geometry.union_all()

# Definir parâmetros para fit da geometria
CRS = 'EPSG:4326'
GRID_SCALE = (0.005, -0.005)

grid_params = helpers.fit_geometry(
    geometry=aoi,
    geometry_crs=CRS,
    grid_crs=CRS,
    grid_scale=GRID_SCALE,
)

# Filtrar a coleção apenas com as bandas de interesse (MSAVI e Albedo)
xavier = xavier.select(['pr', 'ET'])
ds = xr.open_dataset(
    colecao_xavier,
    engine='ee',
    **grid_params,
    chunks='auto',
)


In [14]:
# Calcular indice de aridez (IA) = Precipitação / ETP
ia = ds['pr'] / ds['ET']

In [ ]:
ia.mean(dim=['x', 'y']).plot(figsize=(10, 6), marker='o', color='teal')